# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors (FAIR^2) Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset is defined by a Croissant schema and is accessible at the following URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset (schema and metadata)
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
List available record sets, their fields, and IDs.

In [ ]:
# List all record sets and their fields (by `@id`).

record_sets = list(dataset.recordsets)
if not record_sets:
    print("No record sets detected in Croissant schema. Fetching from underlying objects...")
    # For older/lite schemas, dataset.recordsets may be empty. Try other access if needed.
    # Use dataset.metadata.recordSet (singular/plural varies by schema version)
    if hasattr(metadata, 'recordSet'):
        record_sets = metadata.recordSet if isinstance(metadata.recordSet, list) else [metadata.recordSet]

print(f"Record sets found (by @id):")
for rs in record_sets:
    rs_id = getattr(rs, '@id', None) if hasattr(rs, '@id') else rs.get('@id', None)
    rs_name = getattr(rs, 'name', None) if hasattr(rs, 'name') else rs.get('name', None)
    print(f"- {rs_id or rs}: '{rs_name}'")

    # List fields for this record set
    fields = []
    if hasattr(rs, 'fields'):
        fields = rs.fields
    elif hasattr(rs, 'field'):
        fields = rs.field
    if fields:
        for fld in fields:
            fld_id = getattr(fld, '@id', None) if hasattr(fld, '@id') else fld.get('@id', None)
            fld_name = getattr(fld, 'name', None) if hasattr(fld, 'name') else fld.get('name', None)
            print(f"    Field: {fld_id}: '{fld_name}'")

## 3. Data Extraction
Load data from a specific record set (using record set and field `@id`s from above) into a DataFrame.

In [ ]:
# Extract all record set @id values

# Using dataset.recordsets for Croissant >=1.0; fallback for legacy schemas
rs_objlist = list(dataset.recordsets)
if not rs_objlist and hasattr(metadata, 'recordSet'):
    rs_objlist = metadata.recordSet if isinstance(metadata.recordSet, list) else [metadata.recordSet]
record_set_ids = []
for rs in rs_objlist:
    rs_id = getattr(rs, '@id', None) if hasattr(rs, '@id') else rs.get('@id', None)
    if rs_id is not None:
        record_set_ids.append(rs_id)

if not record_set_ids:
    raise ValueError("No record set @id found in metadata.")

# Load each record set's records as a DataFrame and store by @id
dataframes = {}
for rs_id in record_set_ids:
    try:
        records_iter = dataset.records(record_set=rs_id)
        records = list(records_iter)
        if records:
            dataframes[rs_id] = pd.DataFrame(records)
            print(f"Loaded {len(dataframes[rs_id])} records for {rs_id}")
        else:
            print(f"No records found for {rs_id}")
    except Exception as e:
        print(f"Could not load records for {rs_id}: {e}")

# Pick first record set for detailed exploration
target_rs_id = record_set_ids[0]
print(f"\nRecord set '{target_rs_id}' columns:")
print(dataframes[target_rs_id].columns.tolist())
dataframes[target_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps: filter records, normalize a numeric field, and group/categorize data.

For demonstration, we'll select a numeric column by its `@id` (from the DataFrame columns above) and perform filtering and normalization.

In [ ]:
# Select a numeric column for EDA
df = dataframes[target_rs_id]

# Identify numeric columns (try float/int types)
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if not numeric_cols:
    # Try to cast candidate columns to float
    candidate = None
    for col in df.columns:
        try:
            pd.to_numeric(df[col].dropna().iloc[:5])
            candidate = col
            break
        except:
            continue
    if candidate:
        numeric_cols = [candidate]

print("Numeric fields in first record set:", numeric_cols)

# For this example we'll use the first numeric field
if not numeric_cols:
    raise ValueError("No numeric fields found in the record set for analysis.")
numeric_field = numeric_cols[0]

# Filtering: Show records where value > threshold
threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 10
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records where {numeric_field} > {threshold:.2f}:")
print(filtered_df.head())

# Normalize the selected numeric field (z-score normalization)
filtered_df[f"{numeric_field}_normalized"] = (
    (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
)
print(f"\nNormalized '{numeric_field}' in filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by a categorical field (choose suitable field if available)
categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
group_field = None
if categorical_cols:
    group_field = categorical_cols[0]

if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"\nGrouped data by '{group_field}' (mean {numeric_field}):")
    print(grouped_df.head())

## 5. Visualization
Plot distributions of the selected numeric field, and e.g. boxplot/grouped bar if meaningful.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(8, 5))
sns.histplot(df[numeric_field], bins=15, kde=True)
plt.xlabel(numeric_field)
plt.title(f"Distribution of '{numeric_field}' in Record Set '{target_rs_id}'")
plt.show()

# Boxplot by group if available
if group_field:
    plt.figure(figsize=(8, 5))
    sns.boxplot(x=group_field, y=numeric_field, data=filtered_df)
    plt.title(f"'{numeric_field}' by '{group_field}' (filtered)")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
This notebook has demonstrated how to use the `mlcroissant` library to load metadata and records from a Croissant-defined biomedical dataset, explore available record sets by their `@id`s, and perform initial analysis and visualization. Further analysis can be conducted based on domain-specific research questions. The use of Croissant `@id` for all references ensures robustness and reproducibility across FAIR datasets.

---
Notebook generated for FAIR^2 dataset exploration.